In [ ]:
import transformers
transformers.__version__

In [ ]:
import sys
import numpy as np
import argparse
import pandas as pd
import pickle
import os
import torch
from pathlib import Path
import random
from time import time
import json
import pickle

from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
from huggingface_hub import login
hf_key = "HUGGING_FACE_API_KEY"
login(token=hf_key)
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
seeds = [42, 43, 44]
batch_size = 32
epochs = 0
n_steps = 0
gan_training_epochs = 300


In [ ]:
dataset = 'bank_marketing'

dataset_folder = f"../datasets/{dataset}"

checkpoints_folder = f"{dataset_folder}/checkpoints"
data_folder = f"{dataset_folder}/data"
synthetic_data_folder = f"{dataset_folder}/synthetic-data"

sampling_times = {}


## Great

In [ ]:
from be_great import GReaT

batch_size = 32
epochs = 0

method = 'great'


if method not in sampling_times.keys():
  sampling_times[method] = {}

steps = 13100

llms = [("distilgpt2", "distilgpt2"), ('taptap_distill', 'ztphs980/taptap-distill'), ("qwen3_03B_distil", "tabularisai/Qwen3-0.3B-distil")]

for llm, llm_hugging_face in llms:

  print(f"LLM: {llm}, LLM_Hugging_Face: {llm_hugging_face}")

  sampling_times[method][llm] = {}

  for seed in seeds:
    print(f"Seed: {seed}\n")

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')


    checkpoint_path = f"{checkpoints_folder}/{method}/{llm}/seed{seed}/checkpoint-{steps}"

    model = GReaT(llm=llm_hugging_face,
                experiment_dir=f"{checkpoints_folder}/{method}/{llm}/seed{seed}",
                batch_size=batch_size,
                epochs=epochs,
                save_steps=5000,
                save_total_limit=2)

    trainer = model.fit(train, resume_from_checkpoint = checkpoint_path)

    start = time()

    synthetic_data = model.sample(n_samples=train.shape[0], k=200, max_length=300)

    sampling_time = time() - start

    sampling_times[method][llm][str(seed)] = sampling_time

    synthetic_data.to_csv(f"{synthetic_data_folder}/{method}/{llm}/{dataset_name}_{method}_{llm}_seed{seed}", index=False)



  with open(f"{synthetic_data_folder}/sampling_times.json", "w") as f:
      json.dump(sampling_times, f, indent=2)


## TapTap


In [ ]:
from taptap.taptap import Taptap

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

import lightgbm as lgb

from taptap.taptap import Taptap
from taptap.exp_utils import lightgbm_hpo


In [ ]:
def get_score(train_data, test_data, target_col, best_params):
    train_x = train_data.drop(columns=target_col).copy()
    test_x = test_data.drop(columns=target_col).copy()
    train_y = train_data[[target_col]]
    test_y = test_data[[target_col]]
    train_x, val_x, train_y, val_y = train_test_split(train_x, train_y, test_size=0.2, random_state=1)
    gbm = lgb.LGBMClassifier(**best_params)
    gbm.fit(train_x, train_y, eval_set=[(val_x, val_y)], callbacks=[lgb.early_stopping(50, verbose=False)])
    pred = pd.DataFrame(gbm.predict(test_x), index=test_x.index)
    acc = accuracy_score(test_y, pred)
    return acc, gbm


def make_start_dist(df: pd.DataFrame, col: str):
    s = df[col].dropna()
    if pd.api.types.is_numeric_dtype(s):
        return s.sample(1000, replace=True, random_state=0).astype(str).tolist()
    else:
        vc = s.astype(str).value_counts(normalize=True)
        return vc.to_dict()

In [ ]:
batch_size = 32
n_steps = 0
gradient_accumulation_steps = 1

method = 'taptap'


target_col = "deposit"
task = "classification"

if method not in sampling_times.keys():
  sampling_times[method] = {}


llms = [("distilgpt2", "distilgpt2"), ('taptap_distill', 'ztphs980/taptap-distill'), ("qwen3_03B_distil", "tabularisai/Qwen3-0.3B-distil")]

for llm, llm_hugging_face in llms:

  print(f"LLM: {llm}, LLM_Hugging_Face: {llm_hugging_face}")

  sampling_times[method][llm] = {}

  for seed in seeds:
    print(f"Seed: {seed}\n")

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

    checkpoint_path = f"{checkpoints_folder}/{method}/{llm}/seed{seed}/checkpoint-{training_steps}"

    model = Taptap(llm=llm_hugging_face,
                experiment_dir=f"{checkpoints_folder}/{method}/{llm}/seed{seed}",
                steps=n_steps,
                batch_size=batch_size,
                numerical_modeling='split',
                gradient_accumulation_steps=gradient_accumulation_steps)

    trainer = model.fit(train, target_col=target_col, task=task, resume_from_checkpoint=checkpoint_path)

    col = 'contact'
    dist = make_start_dist(train, col)

    start = time()

    synthetic_data = model.sample(n_samples=train.shape[0],
                                  data=train,
                                  task=task,
                                  max_length=300,
                                  start_col=col,
                                  start_col_dist=dist,
                                  k=200
                                  )

    sampling_time = time() - start

    sampling_times[method][llm][str(seed)] = sampling_time


    categorical_cols = train.select_dtypes(include=["object", "category"]).columns.tolist()

    for col in categorical_cols:
        all_categories = list(
            set(train[col].dropna().unique()) |
            set(test[col].dropna().unique()) |
            set(synthetic_data[col].dropna().unique())
        )
        train[col] = pd.Categorical(train[col], categories=all_categories)
        test[col] = pd.Categorical(test[col], categories=all_categories)
        synthetic_data[col] = pd.Categorical(synthetic_data[col], categories=all_categories)

    best_params = lightgbm_hpo(
      data=train, target_col=target_col, task=task, n_trials=10, n_jobs=16
    )


    acc, gbm = get_score(
      train, test, target_col=target_col, best_params=best_params
    )

    features = synthetic_data.drop(columns=target_col)
    synthetic_data[target_col] = gbm.predict(features)


    synthetic_data.to_csv(f"{synthetic_data_folder}/{method}/{llm}/{dataset_name}_{method}_{llm}_seed{seed}.csv", index=False)


with open(f"{synthetic_data_folder}/sampling_times.json", "w") as f:
    json.dump(sampling_times, f, indent=2)


## Tabula


In [ ]:
from tabula import Tabula

batch_size = 32
epochs = 0

method = 'tabula'

steps = 13100

if method not in sampling_times.keys():
  sampling_times[method] = {}



llms = [("distilgpt2", "distilgpt2"), ('taptap_distill', 'ztphs980/taptap-distill'), ("qwen3_03B_distil", "tabularisai/Qwen3-0.3B-distil")]

for llm, llm_hugging_face in llms:

  print(f"LLM: {llm}, LLM_Hugging_Face: {llm_hugging_face}")

  sampling_times[method][llm] = {}


  for seed in seeds:
    print(f"Seed: {seed}\n")

    set_seed(seed)

    train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
    test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')


    categorical_columns = ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'month', 'poutcome']

    checkpoint_path = f"{checkpoints_folder}/{method}/{llm}/seed{seed}/checkpoint-{steps}"

    model = Tabula(llm=llm_hugging_face,
                  experiment_dir = f"{checkpoints_folder}/{method}/{llm}/seed{seed}",
                  batch_size=batch_size,
                  epochs=epochs,
                  save_steps=5000,
                  save_total_limit=2,
                  categorical_columns = categorical_columns)


    trainer = model.fit(train, resume_from_checkpoint=checkpoint_path)

    start = time()

    synthetic_data = model.sample(n_samples=train.shape[0], k=200, max_length=300)

    sampling_time = time() - start

    sampling_times[method][llm][str(seed)] = sampling_time


    synthetic_data.to_csv(f"{synthetic_data_folder}/{method}/{llm}/{dataset_name}_{method}_{llm}_seed{seed}.csv", index=False)

with open(f"{synthetic_data_folder}/sampling_times.json", "w") as f:
  json.dump(sampling_times, f, indent=2)


## CTGAN

In [ ]:
from sdv.single_table import CTGANSynthesizer

epochs = 0

gan = 'ctgan'

sampling_times[gan] = {}


for seed in seeds:
  print(f"Seed: {seed}\n")

  set_seed(seed)

  model = CTGANSynthesizer.load(f'{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{gan_training_epochs}epochs.pkl')

  train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
  test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

  start = time()

  synthetic_data = model.sample(num_rows=train.shape[0])

  sampling_time = time() - start

  sampling_times[gan][str(seed)] = sampling_time


  synthetic_data.to_csv(f"{synthetic_data_folder}/{gan}/{dataset_name}_{gan}_seed{seed}.csv", index=False)

with open(f"{synthetic_data_folder}/sampling_times.json", "w") as f:
  json.dump(sampling_times, f, indent=2)


## TVAE

In [ ]:
from sdv.single_table import TVAESynthesizer

epochs = 0

gan = 'tvae'

sampling_times[gan] = {}


for seed in seeds:
  print(f"Seed: {seed}\n")

  set_seed(seed)

  model = TVAESynthesizer.load(f'{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{gan_training_epochs}epochs.pkl')

  train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
  test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

  start = time()

  synthetic_data = model.sample(num_rows=train.shape[0])

  sampling_time = time() - start

  sampling_times[gan][str(seed)] = sampling_time

  synthetic_data.to_csv(f"{synthetic_data_folder}/{gan}/{dataset_name}_{gan}_seed{seed}.csv", index=False)

with open(f"{synthetic_data_folder}/sampling_times.json", "w") as f:
  json.dump(sampling_times, f, indent=2)


## TabFairGAN

In [ ]:
epochs = 0

gan = 'tabfairgan'

sampling_times[gan] = {}


for seed in seeds:
  print(f"Seed: {seed}\n")

  set_seed(seed)

  with open(f'{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{gan_training_epochs}epochs.pkl', 'rb') as f:
    model = pickle.load(f)

  train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
  test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

  start = time()

  synthetic_data = model.generate_fake_df(num_rows=train.shape[0])

  sampling_time = time() - start

  sampling_times[gan][str(seed)] = sampling_time


  synthetic_data.to_csv(f"{synthetic_data_folder}/{gan}/{dataset_name}_{gan}_seed{seed}.csv", index=False)

with open(f"{synthetic_data_folder}/sampling_times.json", "w") as f:
  json.dump(sampling_times, f, indent=2)


## CTAB-GAN

In [ ]:
from model.ctabgan import CTABGAN

epochs = 0

gan = 'ctabgan'

sampling_times[gan] = {}

for seed in seeds:
  print(f"Seed: {seed}\n")

  set_seed(seed)

  with open(f'{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{gan_training_epochs}epochs.pkl', 'rb') as f:
    model = pickle.load(f)

  train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
  test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

  start = time()

  synthetic_data = model.generate_samples()

  sampling_time = time() - start

  sampling_times[gan][str(seed)] = sampling_time


  synthetic_data.to_csv(f"{synthetic_data_folder}/{gan}/{dataset_name}_{gan}_seed{seed}.csv", index=False)

with open(f"{synthetic_data_folder}/sampling_times.json", "w") as f:
  json.dump(sampling_times, f, indent=2)


## CTAB-GAN-Plus

In [ ]:
from model.ctabgan import CTABGAN

epochs = 0

gan = 'ctabgan_plus'

sampling_times[gan] = {}

for seed in seeds:
  print(f"Seed: {seed}\n")

  set_seed(seed)

  with open(f'{checkpoints_folder}/{gan}/{dataset_name}_{gan}_seed{seed}_{gan_training_epochs}epochs.pkl', 'rb') as f:
    model = pickle.load(f)

  train = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_train.csv')
  test = pd.read_csv(f'{data_folder}/{dataset_name}_seed{seed}_test.csv')

  start = time()

  synthetic_data = model.generate_samples()

  sampling_time = time() - start

  sampling_times[gan][str(seed)] = sampling_time


  synthetic_data.to_csv(f"{synthetic_data_folder}/{gan}/{dataset_name}_{gan}_seed{seed}.csv", index=False)

with open(f"{synthetic_data_folder}/sampling_times.json", "w") as f:
  json.dump(sampling_times, f, indent=2)
